# Infant Rest/Movie Analysis: ISC and IDE Results


In [ ]:

import numpy as np
import pandas as pd
import os, sys, glob
import matplotlib.pyplot as plt
import seaborn as sns
import nibabel as nib
from nilearn import plotting, image
import nilearn
from scipy.spatial.distance import dice
from nilearn.maskers import NiftiMasker
import infant_restmovie_utils as iru
import infant_restmovie_config as irc
import adult_restmovie_utils as aru
import adult_restmovie_config as arc
import plotting_helpers as helper
from matplotlib.colors import ListedColormap
import stats_helpers as sh
from obspy.imaging.cm import viridis_white
from scipy import stats

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# Load average results for aeronaut and mickey tasks
results_dir = iru.get_results_dir()
print(f"Results directory: {results_dir}")

# Define tasks and measures
movie_tasks = ['aeronaut', 'sleep']
measures = {'ISC': 'ISC', 'TPHATE_DiffOp_IDE': 'IDE'}

# Load the average result volumes
avg_results = {}
all_results = {}
avg_imgs = {}
for task in movie_tasks:
    avg_results[task] = {}
    all_results[task] = {}
    avg_imgs[task] = {}
    mask_file = iru.get_intersect_mask(task)
    masker = NiftiMasker(mask_img=mask_file)
    for measure in measures.keys():
        fn = f'{results_dir}/{task}_{measure}_all_subjects_results.nii.gz'
        if os.path.exists(fn):
            img = nib.load(fn)
            # average over subjects - across 4th dimension
            data = masker.fit_transform(img)
            mean_img = np.nanmean(data, axis=0)
            avg_results[task][measure] = mean_img
            mean_nii = masker.inverse_transform(mean_img)
            avg_imgs[task][measure] = mean_nii
            all_results[task][measure] = data
            print(f"Loaded {task} {measure}: {avg_results[task][measure].shape}")
            print(f"Loaded {task} {measure}: {all_results[task][measure].shape}")
            print(f"Loaded {task} {measure}: {avg_imgs[task][measure].shape}")
            
        else:
            print(f"Warning: {fn} not found")


In [ ]:

# Define tasks and measures
movie_tasks = ['aeronaut' ]
measures = {'ISC': 'ISC', 'TPHATE_DiffOp_IDE': 'IDE'}
masker=iru.get_intersect_mask('aeronaut')
masker = NiftiMasker(mask_img=masker)
# Load the average result volumes
adult_avg_results = {}
for task in movie_tasks:
    adult_avg_results[task] = {}
    for measure in measures.keys():
        fn = f'adult_restmovie/results/{task}_{measure}_all_subjects_results.nii.gz'
        if os.path.exists(fn):
            nii = nib.load(fn)
            adult_avg_results[task][measure] = masker.fit_transform(nii)
            print(f"Loaded {task} {measure}: {adult_avg_results[task][measure].shape}; mean: {np.nanmean(adult_avg_results[task][measure]):.3f}")
        else:
            print(f"Warning: {fn} not found")



In [ ]:
np.nanmean(all_results['aeronaut']['TPHATE_DiffOp_IDE']),np.nanstd(np.nanmean(all_results['aeronaut']['TPHATE_DiffOp_IDE'],axis=0)), np.nanmean(adult_avg_results['aeronaut']['TPHATE_DiffOp_IDE']),np.nanstd(np.nanmean(adult_avg_results['aeronaut']['TPHATE_DiffOp_IDE'],axis=0))

In [ ]:
stats.ttest_ind(np.nanmean(all_results['aeronaut']['TPHATE_DiffOp_IDE'], axis=1), np.nanmean(adult_avg_results['aeronaut']['TPHATE_DiffOp_IDE'], axis=1))

In [ ]:
# Prepare data for surface plotting grid
# We'll create a 2x1 grid: aeronaut ISC, aeronaut IDE
nifti_images = []
titles = []

for task in ['aeronaut']:
    for measure, label in measures.items():
        if measure in avg_results[task]:
            # inverse masker to get nifti image
            nifti_images.append(avg_imgs[task][measure])
            titles.append(f'{task.capitalize()} {label}')
nifti_images.append(avg_imgs['sleep']['TPHATE_DiffOp_IDE']
                    )
titles.append('Sleep IDE')

# Determine colorbar ranges
# ISC typically ranges from 0 to ~0.5
# IDE typically ranges from ~1 to ~22
cbar_ranges = [(0, 0.4), (1, 18), (1, 25)]
cmaps = ['magma', viridis_white, viridis_white]
cbar_labels = ['ISC', 'IDE','IDE']

print(f"Prepared {len(nifti_images)} images for visualization")
print(f"Titles: {titles}")


In [ ]:
# Create the surface plot grid for movie tasks (aeronaut and mickey)
output_path = os.path.join('main_plots/infant_ISC_IDE_surface_grid.pdf')

# Generate individual surface plots
temp_fns = []
for idx, (img, title, cmap, cbar_range) in enumerate(zip(nifti_images, titles, cmaps, cbar_ranges)):
    temp_fn = f'/tmp/infant_plot_{idx}.png'
    
    helper.generate_surface_plot(
        data_fn=img, 
        image_fn=temp_fn,
        atlas='searchlight', 
        cmap=cmap, 
        cbar_range=cbar_range,
        surf_type='fslr', 
        target_density='32k',
        include_cbar=True, 
        title=title,
        method='linear', 
        threshold=None, 
        mask_medial_wall=True
    )
    temp_fns.append(temp_fn)

# Compile all surface plots into a grid
helper.compile_surface_plots_to_grid(
    image_files=temp_fns,
    atlas='searchlight',
    data_files=nifti_images,
    surf_type='fslr',
    target_density='32k',
    output_path=output_path,
    main_title='infant'
)

In [ ]:
d1 = nib.load('infant_restmovie/results/task_difference_maps/sleep_aeronaut_two_samp_ttest_output_tfce_corrp_tstat1.nii.gz')
mask = image.math_img('np.where(X >= 0.95, 1, 0)', X=d1)
mean_img = nib.load('infant_restmovie/results/task_difference_maps/aeronaut_sleep_TPHATE_DiffOp_IDE_SL_difference_maps_avg_unthresholded.nii.gz')
masked_mean_img = image.math_img('X * Y', X=mean_img, Y=mask)
helper.generate_surface_plot(data_fn=masked_mean_img, 
                             image_fn=None, 
                             atlas='searchlight', 
                             surf_type='fslr',
                            target_density='32k',
                            method='linear',
                             cmap=helper.diverging_colormap_bp(),
                             cbar_range=(-11,11), 
                             include_cbar=True)



In [ ]:
isc_img = avg_imgs.get('aeronaut', {}).get('ISC', None)
if isc_img is None:
    raise RuntimeError("Aeronaut ISC image not loaded. Ensure section 1 loaded 'aeronaut' ISC average results.")

diff_img = masked_mean_img 

isc_data = isc_img.get_fdata()
diff_data = diff_img.get_fdata()

# Basic shape check
if isc_data.shape != diff_data.shape:
    raise ValueError(f"Shape mismatch: ISC {isc_data.shape} vs diff {diff_data.shape}")

# --- Top 10% ISC mask (ignore NaNs) ---
isc_valid = isc_data[~np.isnan(isc_data)]
if isc_valid.size == 0:
    raise RuntimeError("ISC image contains only NaNs.")
threshold=95
isc_thr = np.nanpercentile(isc_valid, threshold)
top_mask = (isc_data >= isc_thr) & ~np.isnan(isc_data)

In [ ]:
# Layer 0: the thresholded Rest−Aeronaut IDE difference map (diverging colormap)
# Layer 1: a binary/top-percentile ISC mask (yellow overlay)
diff_img = masked_mean_img
threshold = 95  # top 5% ISC
top_mask_vol = np.full(diff_img.shape, np.nan, dtype=float)
top_mask_vol[top_mask] = 1.0
top_mask_img = nib.Nifti1Image(top_mask_vol, affine=diff_img.affine, header=diff_img.header)

helper.generate_surface_plot(
    data_fn=[diff_img, top_mask_img],
    image_fn="main_plots/infant_aeronaut_sleep_thresholded_with_top_ISC_overlay.pdf",  # saves to 
    atlas='searchlight',
    cmap=helper.diverging_colormap_bp(),
    cbar_range=(-11, 11),
    surf_type='fslr',
    target_density='32k',
    include_cbar=True,  # broadcast; overridden per-layer below
    title='Rest − Aeronaut IDE (thresholded) + Top ISC voxels',
    method='linear',
    threshold=None,
    alpha=0.5,
    mask_medial_wall=True,
    layers_kwargs=[
        dict(
            cmap=helper.diverging_colormap_bp(),
            cbar_range=(-11, 11),
            alpha=1,
            label='ΔIDE (Rest − Aeronaut)',
            cbar=True,
        ),
        dict(
            cmap=ListedColormap([(1,1,0.6,1.0)]), 
            cbar_range=(0, 1),
            alpha=0.6,
            label=f'Top {100 - threshold}% ISC mask',
            cbar=True,
        ),
    ],
)

In [ ]:
# Compute the dice coefficient between top ISC mask and significant IDE difference voxels
significant_diff_mask = masker.transform(mask) > 0  # Assuming positive differences are significant
top_isc_mask = masker.transform(top_mask_img) > 0
sh.overlap_coefficient(significant_diff_mask.ravel(), top_isc_mask.ravel())



## 2. Visualize TPHATE_DiffOp_IDE for sleep (resting state)


In [ ]:
# Load sleep task IDE results
sleep_ide_fn = f'{results_dir}/sleep_TPHATE_DiffOp_IDE_average_results.nii.gz'

if os.path.exists(sleep_ide_fn):
    sleep_ide_img = nib.load(sleep_ide_fn)
    print(f"Loaded sleep IDE: {sleep_ide_img.shape}")
    
    # Generate surface plot for sleep IDE
    output_fn = os.path.join(results_dir, 'sleep_TPHATE_DiffOp_IDE_surface.png')
    
    helper.generate_surface_plot(
        data_fn=sleep_ide_img,
        image_fn=output_fn,
        atlas='searchlight',
        cmap='viridis',
        cbar_range=(2, 20),
        surf_type='fsaverage',
        target_density='41k',
        include_cbar=True,
        title='Sleep (Resting State) IDE',
        method='nearest',
        threshold=None,
        mask_medial_wall=True
    )
    
    # Display the plot
    if os.path.exists(output_fn):
        img_array = plt.imread(output_fn)
        fig, ax = plt.subplots(figsize=(15, 8))
        ax.imshow(img_array)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print(f"Warning: {sleep_ide_fn} not found")



## 3. Task Difference Maps: Sleep - Aeronaut and Sleep - Mickey

These maps show regions where IDE differs significantly between sleep (resting state) and movie viewing tasks, with FDR correction for multiple comparisons.


In [ ]:

# Load the task difference maps (sleep - movie tasks)
diff_maps_dir = f'{results_dir}/task_difference_maps'
task_pairs = ['aeronaut_sleep', 'mickey_sleep']

# Storage for difference maps
diff_results = {}

for pair in task_pairs:
    diff_results[pair] = {}
    
    # Load thresholded (FDR-corrected) difference map
    thresh_fn = f'{diff_maps_dir}/{pair}_TPHATE_DiffOp_IDE_SL_difference_maps_avg_thresholded.nii.gz'
    if os.path.exists(thresh_fn):
        diff_results[pair]['thresholded'] = nib.load(thresh_fn)
        print(f"Loaded {pair} thresholded map: {diff_results[pair]['thresholded'].shape}")
    else:
        print(f"Warning: {thresh_fn} not found")
    
    # Load unthresholded difference map for reference
    unthresh_fn = f'{diff_maps_dir}/{pair}_TPHATE_DiffOp_IDE_SL_difference_maps_avg_unthresholded.nii.gz'
    if os.path.exists(unthresh_fn):
        diff_results[pair]['unthresholded'] = nib.load(unthresh_fn)
        print(f"Loaded {pair} unthresholded map: {diff_results[pair]['unthresholded'].shape}")
    
    # Load adjusted p-values
    pval_fn = f'{diff_maps_dir}/{pair}_TPHATE_DiffOp_IDE_SL_difference_maps_adj_pvals_indep_samp_ttest.nii.gz'
    if os.path.exists(pval_fn):
        diff_results[pair]['pvals'] = nib.load(pval_fn)
        print(f"Loaded {pair} adjusted p-values: {diff_results[pair]['pvals'].shape}")


In [ ]:
diff_results[task_pairs[0]]

In [ ]:


# Determine appropriate colorbar range for difference maps
# These are differences, so we want a diverging colormap centered at 0

# Check the range of values in the thresholded maps
diff_ranges = []
for pair in task_pairs:
    if 'thresholded' in diff_results[pair]:
        data = diff_results[pair]['thresholded'].get_fdata()
        # Remove NaN values for range calculation
        valid_data = data[~np.isnan(data)]
        if len(valid_data) > 0:
            vmin, vmax = np.nanmin(valid_data), np.nanmax(valid_data)
            diff_ranges.append((vmin, vmax))
            print(f"{pair} range: {vmin:.3f} to {vmax:.3f}")

# Use symmetric range for diverging colormap
if diff_ranges:
    abs_max = max(abs(r[0]) for r in diff_ranges + [(0, 0)] if r[0] is not None) 
    abs_max = max(abs_max, max(abs(r[1]) for r in diff_ranges + [(0, 0)] if r[1] is not None))
    cbar_range_diff = (-abs_max, abs_max)
    print(f"Using symmetric colorbar range: {cbar_range_diff}")
else:
    cbar_range_diff = (-5, 5)
    print("Using default colorbar range: (-5, 5)")

# Get diverging colormap
div_cmap = helper.diverging_colormap_bp()

# Visualize sleep - aeronaut difference map (FDR-thresholded)
pair = 'aeronaut_sleep'
if 'thresholded' in diff_results[pair]:
    output_fn = os.path.join(results_dir, f'{pair}_IDE_difference_thresholded_surface.png')
    
    helper.generate_surface_plot(
        data_fn=diff_results[pair]['thresholded'],
        image_fn=output_fn,
        atlas='searchlight',
        cmap=div_cmap,
        cbar_range=cbar_range_diff,
        surf_type='fsaverage',
        target_density='41k',
        include_cbar=True,
        title='Sleep - Aeronaut IDE (FDR corrected)',
        method='nearest',
        threshold=None,  # Already thresholded
        mask_medial_wall=True
    )
    
    # Display the plot
    if os.path.exists(output_fn):
        img_array = plt.imread(output_fn)
        fig, ax = plt.subplots(figsize=(15, 8))
        ax.imshow(img_array)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print(f"Saved to: {output_fn}")



In [ ]:

# Visualize sleep - mickey difference map (FDR-thresholded)
pair = 'mickey_sleep'
if 'thresholded' in diff_results[pair]:
    output_fn = os.path.join(results_dir, f'{pair}_IDE_difference_thresholded_surface.png')
    
    helper.generate_surface_plot(
        data_fn=diff_results[pair]['thresholded'],
        image_fn=output_fn,
        atlas='searchlight',
        cmap=div_cmap,
        cbar_range=cbar_range_diff,
        surf_type='fsaverage',
        target_density='41k',
        include_cbar=True,
        title='Sleep - Mickey IDE (FDR corrected)',
        method='nearest',
        threshold=None,  # Already thresholded
        mask_medial_wall=True
    )
    
    # Display the plot
    if os.path.exists(output_fn):
        img_array = plt.imread(output_fn)
        fig, ax = plt.subplots(figsize=(15, 8))
        ax.imshow(img_array)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print(f"Saved to: {output_fn}")

# Create a combined figure showing both difference maps side by side
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

for idx, pair in enumerate(task_pairs):
    if 'thresholded' in diff_results[pair]:
        temp_fn = f'/tmp/diff_plot_{pair}.png'
        
        task_name = pair.split('_')[0].capitalize()
        title = f'Sleep - {task_name} IDE (FDR corrected)'
        
        helper.generate_surface_plot(
            data_fn=diff_results[pair]['thresholded'],
            image_fn=temp_fn,
            atlas='searchlight',
            cmap=div_cmap,
            cbar_range=cbar_range_diff,
            surf_type='fsaverage',
            target_density='41k',
            include_cbar=True,
            title=title,
            method='nearest',
            threshold=None,
            mask_medial_wall=True
        )
        
        if os.path.exists(temp_fn):
            img_array = plt.imread(temp_fn)
            axes[idx].imshow(img_array)
            axes[idx].axis('off')
            axes[idx].set_title(title, fontsize=14, fontweight='bold')

plt.suptitle('Task Differences: Resting State vs Movie Viewing (IDE)', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()

output_combined = os.path.join(results_dir, 'sleep_vs_movies_IDE_differences_combined.png')
plt.savefig(output_combined, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved combined difference maps to: {output_combined}")



## 4. Summary Statistics

Let's examine some summary statistics about the differences between conditions.


In [ ]:

# Calculate summary statistics for each difference map
print("=" * 80)
print("SUMMARY STATISTICS FOR TASK DIFFERENCES")
print("=" * 80)

for pair in task_pairs:
    print(f"\n{pair.upper().replace('_', ' - ')}:")
    print("-" * 80)
    
    if 'thresholded' in diff_results[pair]:
        # Thresholded data
        thresh_data = diff_results[pair]['thresholded'].get_fdata()
        valid_thresh = thresh_data[~np.isnan(thresh_data)]
        
        if len(valid_thresh) > 0:
            print(f"  Thresholded (FDR-corrected) differences:")
            print(f"    Number of significant voxels: {len(valid_thresh)}")
            print(f"    Mean difference: {np.mean(valid_thresh):.3f}")
            print(f"    Std difference: {np.std(valid_thresh):.3f}")
            print(f"    Range: [{np.min(valid_thresh):.3f}, {np.max(valid_thresh):.3f}]")
            
            # Count positive and negative differences
            n_pos = np.sum(valid_thresh > 0)
            n_neg = np.sum(valid_thresh < 0)
            print(f"    Positive differences (sleep > movie): {n_pos} voxels")
            print(f"    Negative differences (sleep < movie): {n_neg} voxels")
        else:
            print(f"  No significant differences found (after FDR correction)")
    
    if 'unthresholded' in diff_results[pair]:
        # Unthresholded data for comparison
        unthresh_data = diff_results[pair]['unthresholded'].get_fdata()
        valid_unthresh = unthresh_data[~np.isnan(unthresh_data)]
        
        if len(valid_unthresh) > 0:
            print(f"\n  Unthresholded differences (all voxels):")
            print(f"    Number of voxels: {len(valid_unthresh)}")
            print(f"    Mean difference: {np.mean(valid_unthresh):.3f}")
            print(f"    Std difference: {np.std(valid_unthresh):.3f}")
    
    if 'pvals' in diff_results[pair]:
        # P-value statistics
        pval_data = diff_results[pair]['pvals'].get_fdata()
        valid_pvals = pval_data[~np.isnan(pval_data)]
        
        if len(valid_pvals) > 0:
            n_sig_05 = np.sum(valid_pvals < 0.05)
            n_sig_01 = np.sum(valid_pvals < 0.01)
            n_sig_001 = np.sum(valid_pvals < 0.001)
            
            print(f"\n  Significance levels (FDR-adjusted p-values):")
            print(f"    p < 0.05: {n_sig_05} voxels ({100*n_sig_05/len(valid_pvals):.2f}%)")
            print(f"    p < 0.01: {n_sig_01} voxels ({100*n_sig_01/len(valid_pvals):.2f}%)")
            print(f"    p < 0.001: {n_sig_001} voxels ({100*n_sig_001/len(valid_pvals):.2f}%)")

print("\n" + "=" * 80)


In [ ]:
par_df = pd.read_csv('infant_restmovie/participant_info.csv')
par_df.head()

In [ ]:
results_df = pd.read_csv('infant_restmovie/results/parcelwise_results_ISC_IDE.csv', index_col=0)
df_corr = pd.DataFrame(columns=['subject','rho','pval','zscore','age'])
par_df = pd.read_csv('infant_restmovie/participant_info.csv')
par_df = par_df[par_df['task']=='aeronaut']
task='aeronaut'
results_df = results_df[results_df['task']==task]
for s in results_df['subject'].unique():
    temp = results_df[results_df['subject'] == s]
    if len(temp) < 400 :
        continue
    ide_movie = temp[temp['measure']=='TPHATE_DiffOp_IDE']['score'].values
    isc = temp[temp['measure']=='ISC']['score'].values
    pval, obs, zsc = sh.permute_pattern(ide_movie, isc, n_permutations=1000, random_state=None)
    age = par_df[par_df['subject_id']==s]['age'].values[0]
    df_corr.loc[len(df_corr)] = [s, obs, pval, zsc, age
                            ]

In [ ]:
from scipy import stats

df_corr = pd.DataFrame(columns=['subject','rho','pval','zscore','age'])
par_df = pd.read_csv('infant_restmovie/participant_info.csv')
task='aeronaut'
par_df=par_df[par_df['task']==task].reset_index(drop=True)

# Load subject-level ISC and IDE data
isc_fn = f'infant_restmovie/results/{task}_ISC_all_subjects_results.nii.gz'
ide_fn = f'infant_restmovie/results/{task}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'

if os.path.exists(isc_fn) and os.path.exists(ide_fn):
    isc_img = nib.load(isc_fn)
    ide_img = nib.load(ide_fn)
    mask = nib.load(irc.INFANT_INTERSECT_MASKS[task])
    masker = NiftiMasker(mask_img=mask).fit(mask)
    # now apply the task-relevant intersection mask to both
    isc_data = masker.fit_transform(isc_img)
    ide_data = masker.fit_transform(ide_img)
    
    # Calculate voxel-wise correlation across subjects
    # For each voxel, correlate ISC values across subjects with IDE values across subjects
    results_map = np.zeros((isc_data.shape[0],2))
    for i in range(isc_data.shape[0]):
        pval, obs, zsc = sh.permute_pattern(ide_data[i], isc_data[i], n_permutations=1000, random_state=None)
        par,age=par_df.loc[i,['subject_id','age']]
        df_corr.loc[len(df_corr)] = [par, obs, pval, zsc, age]


In [ ]:
df_corr.to_csv('infant_restmovie/results/isc_ide_correlation_null_stats.csv')

In [ ]:
sns.barplot(data=df_corr,  y='zscore')
sns.stripplot(data=df_corr,  y='zscore')


In [ ]:
results_df.groupby(['task','measure','region_name']).mean()

In [ ]:
spearman_rho, pval = scipy.stats.spearmanr(df_corr['age'], df_corr['zscore'])
print(f"Spearman correlation between age and ISC-IDE correlation z-score: rho={spearman_rho:.3f}, p={pval:.3e}")

In [ ]:
# Plot these results as a bar plot with error bars
plt.figure(figsize=(4,6))
sns.barplot(data=res_df, x='task', y='zscore',palette=plotting_helpers.get_paired_palette()[2:], edgecolor='k', linewidth=2, alpha=0.8)
sns.stripplot(data=res_df, x='task', y='spearman_rho',palette=plotting_helpers.get_paired_palette()[2:], edgecolor='k', linewidth=1, size=10, jitter=True)
# Run t-tests to compare against zero
for task in res_df['task'].unique():
    task_data = res_df[res_df['task'] == task]['spearman_rho'].dropna()
    t_stat, p_val = stats.ttest_1samp(task_data, 0)
    print(f"T-test for {task}: t={t_stat:.3f}, p={p_val:.4f}")  
# plot significance asterisks

y_max = res_df['spearman_rho'].max() 
for idx, task in enumerate(res_df['task'].unique()):
    task_data = res_df[res_df['task'] == task]['spearman_rho'].dropna()
    t_stat, p_val = stats.ttest_1samp(task_data, 0)
    if p_val < 0.001:
        sig = '***'
    elif p_val < 0.01:
        sig = '**'
    elif p_val < 0.05:
        sig = '*'
    else:
        sig = 'n.s.'
    plt.text(idx, 0.5, sig, ha='center', va='bottom', fontsize=16)

plt.ylim(y_min := res_df['spearman_rho'].min() - 0.1, y_max + 0.25)
plt.title('Brain-wide correlation between ISC and IDE', fontsize=16)
plt.ylabel('Spearman Correlation (rho)', fontsize=14)
plt.xlabel('Movie Task', fontsize=14)
plt.axhline(0, color='k', linestyle='--')
sns.despine()

In [ ]:
res_df.to_csv(f'{results_dir}/infant_movie_tasks_ISC_IDE_correlation_results.csv', index=False)

In [ ]:
# Plot the average ID across subjects for the aeronaut, mickey, and sleep tasks
average_ides = {}
for task in ['aeronaut', 'mickey', 'sleep']:
    ide_fn = f'infant_restmovie/results/{task}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'
    if os.path.exists(ide_fn):
        ide_img = nib.load(ide_fn)
        data = ide_img.get_fdata()
        mean_data = np.nanmean(data, axis=3)
        mean_img = nib.Nifti1Image(mean_data, ide_img.affine, ide_img.header)
        average_ides[task] = mean_img
        print(f"Loaded {task} IDE: {average_ides[task].shape}; mean: {np.nanmean(average_ides[task].get_fdata())}")
    else:
        print(f"Warning: {ide_fn} not found")
# Plot average IDEs
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
tasks = ['aeronaut', 'mickey', 'sleep']
for idx, task in enumerate(tasks):
    if task in average_ides:
        temp_fn = f'/tmp/infant_avg_ide_{task}.png'
        helper.generate_surface_plot(
            data_fn=average_ides[task],
            image_fn=temp_fn,
            atlas='searchlight',
            cmap='viridis',
            cbar_range=(2, 20),
            surf_type='fsaverage',
            target_density='41k',
            include_cbar=True,
            title=f'Average IDE: {task.capitalize()}',
            method='nearest',
            threshold=None,
            mask_medial_wall=True
        )
        if os.path.exists(temp_fn):
            img_array = plt.imread(temp_fn)
            axes[idx].imshow(img_array)
            axes[idx].axis('off')
            axes[idx].set_title(f'Average IDE: {task.capitalize()}', fontsize=14, fontweight='bold')

In [ ]:
from scipy.stats import ttest_rel, ttest_ind
results_dir = iru.get_results_dir()
# Load subject-level IDE data for all three tasks and compute voxel-wise averages per subject
ide_subject_averages = []

for task in ['aeronaut', 'mickey', 'sleep']:
    ide_fn = f'{results_dir}/{task}_TPHATE_DiffOp_IDE_all_subjects_results.nii.gz'
    
    if os.path.exists(ide_fn):
        ide_img = nib.load(ide_fn)
        ide_data = ide_img.get_fdata()
        
        # Get number of subjects (4th dimension)
        n_subjects = ide_data.shape[3]
        
        # For each subject, compute the mean IDE across all voxels
        for subj_idx in range(n_subjects):
            subj_data = ide_data[:, :, :, subj_idx]
            # Compute mean, ignoring NaN values
            mean_ide = np.nanmean(subj_data)
            
            ide_subject_averages.append({
                'task': task,
                'subject_id': subj_idx,
                'mean_IDE': mean_ide
            })
        
        print(f"Loaded {task}: {n_subjects} subjects")
    else:
        print(f"Warning: {ide_fn} not found")


In [ ]:

# Create DataFrame
ide_df = pd.DataFrame(ide_subject_averages)
print(f"\nCreated DataFrame with {len(ide_df)} rows")
print(ide_df.head())

# Create barplot
plt.figure(figsize=(6,8))
sns.barplot(data=ide_df, x='task', y='mean_IDE', 
            palette=helper.get_paired_palette()[2:], 
            edgecolor='k', linewidth=2, alpha=0.8)
sns.stripplot(data=ide_df, x='task', y='mean_IDE', 
              palette=helper.get_paired_palette()[2:],
                size=12, alpha=1, jitter=True, edgecolor='k', linewidth=1)

# Add statistical comparisons

# Compare pairs of tasks
task_pairs_to_compare = [('aeronaut', 'mickey'), ('aeronaut', 'sleep'), ('mickey', 'sleep')]

y_max = ide_df['mean_IDE'].max()
y_min = ide_df['mean_IDE'].min()

for idx, (task1, task2) in enumerate(task_pairs_to_compare):
    data1 = ide_df[ide_df['task'] == task1]['mean_IDE'].values
    data2 = ide_df[ide_df['task'] == task2]['mean_IDE'].values
    
    # Use independent samples t-test since different tasks may have different subjects
    t_stat, p_val = ttest_ind(data1, data2)
    
    print(f"\nT-test {task1} vs {task2}: t={t_stat:.3f}, p={p_val:.4f}")

# Plot pairwise significance
for idx, (task1, task2) in enumerate(task_pairs_to_compare):
    data1 = ide_df[ide_df['task'] == task1]['mean_IDE'].values
    data2 = ide_df[ide_df['task'] == task2]['mean_IDE'].values
    
    t_stat, p_val = ttest_ind(data1, data2)
    
    # Determine significance level
    if p_val < 0.001:
        sig = '***'
    elif p_val < 0.01:
        sig = '**'
    elif p_val < 0.05:
        sig = '*'
    else:
        sig = 'n.s.'
    
    # Determine y position for the annotation
    y, h, col = y_max + 0.2 + idx*0.4, 0.05, 'k'
    x1, x2 = tasks.index(task1), tasks.index(task2)
    plt.plot([x1, x2], [y,  y], lw=1.5, c=col)
    plt.text((x1 + x2) * .5, y + h, sig, ha='center', va='bottom', color=col, fontsize=14)

plt.ylabel('Mean IDE (across voxels)', fontsize=14)
plt.xlabel('Task', fontsize=14)
plt.title('Average IDE by Task', fontsize=16)
sns.despine()
plt.tight_layout()


In [ ]:
import parcelwise_regressions as pwr
results_df = pd.read_csv('infant_restmovie/results/parcelwise_results_ISC_IDE.csv', index_col=0)
results_df=results_df[results_df['task']=='aeronaut']
REGION_ORDER = results_df['region_name'].values[:400]
# Pivot the results dataframe to have TPHATE_DiffOp and ISC as columns
results_df1 = results_df.pivot_table(index=['subject','region_name','task'], 
                                      columns='measure', values='score').reset_index()
results_df1['age']=[par_df[par_df['subject_id']==s]['age'].values[0] for s in results_df1['subject'].values]
task='aeronaut'


In [ ]:
results_df1

In [ ]:
sig_mask

In [ ]:
# formula = f"TPHATE_DiffOp_IDE ~ ISC + age"

# a = sh.parcelwise_regression(results_df1, "age", yname="TPHATE_DiffOp_IDE", formula=formula, 
#                          region_order=REGION_ORDER, alpha=0.05, fdr_method='fdr_bh')
# a = a.set_index('region_name').reindex(REGION_ORDER).reset_index()

# helper.generate_surface_plot( 
#                 a['r2'].values,
#                 image_fn=None,#"main_plots/HBN_age_predicts_ide_movieTP_r2_masked.pdf",
#                 atlas='Schaefer',
#                 cmap='magma',
#                 cbar_range=[0,0.35],
#                 surf_type='fslr',
#                 target_density='32k',
#                 method='linear',
#                 include_cbar=True,
#                 title=rf'Model R^2',
#                 threshold=None,
#                 mask_medial_wall=True
#             )

to_plot = ['age', 'ISC']
cmaps = [helper.diverging_colormap_gpu(),helper.diverging_colormap_gpu(), 'magma']
for var,cmap,thr in zip(to_plot, cmaps, [None, 4]):
    sig_mask = a.set_index('region_name').reindex(REGION_ORDER)[f'sig_{var}_fdr'].values
    print(f"Plotting {var} with {np.sum(sig_mask)} significant regions")
    #sig_mask = np.ones_like(a.set_index('region_name').reindex(REGION_ORDER)[f'coef_{var}'].values)
    coef_masked = a.set_index('region_name').reindex(REGION_ORDER)[f'coef_{var}'].values*sig_mask
    t_mask = a.set_index('region_name').reindex(REGION_ORDER)[f'tstat_{var}'].values * sig_mask

    cbar_range, this_cmap = helper.determine_colorbar_range(coef_masked, cmap)
    
    helper.generate_surface_plot( 
                coef_masked,
                image_fn=f"main_plots/infants_age_predicts_ide_{var}_coef_masked.pdf",
                atlas='Schaefer',
                cmap=this_cmap,
                cbar_range=cbar_range,
                surf_type='fslr',
                target_density='32k',
                method='linear',
                include_cbar=True,
                title=rf'$\beta$_{var}',
                threshold=None,
                mask_medial_wall=True
            )
    
    # cbar_range, this_cmap = helper.determine_colorbar_range(t_mask, cmap)
    # helper.generate_surface_plot( 
    #             t_mask,
    #             image_fn=None,#f"main_plots/HBN_age_predicts_ide_movieTP_{var}_tstat_masked.pdf",
    #             atlas='Schaefer',
    #             cmap=this_cmap,
    #             cbar_range=cbar_range,
    #             surf_type='fslr',
    #             target_density='32k',
    #             method='linear',
    #             include_cbar=True,
    #             title=rf'tstat_{var}',
    #             threshold=None,
    #             mask_medial_wall=True
    #         )

In [ ]:
# Load in data for IDE sleep and movies, average across voxels within participant, and plot as barplot